In [ ]:
import os
import numpy as np
import cupy as cp
import h5py

import twixtools
import pickle
import sigpy as sp
from sigpy.mri import app, cc, nlrecon, nlop
from pathlib import Path
from scipy.ndimage import convolve
from RadialSlidingRecon.matlab_functions_in_python import *
from RadialSlidingRecon.utility import *
import argparse


from time import time

import yaml
from yaml import Loader
from scipy.io import loadmat

import sys
from pprint import pprint
import matplotlib.pyplot as plt
import matplotlib.patches as patches
plt.rcParams.update({
    "font.size": 16,
    "font.family": "serif",       # or "sans-serif", "monospace"
    "font.serif": ["Times New Roman"],
})

from matplotlib.colors import LinearSegmentedColormap

cm_data = [[0.2081, 0.1663, 0.5292], [0.2116238095, 0.1897809524, 0.5776761905], 
 [0.212252381, 0.2137714286, 0.6269714286], [0.2081, 0.2386, 0.6770857143], 
 [0.1959047619, 0.2644571429, 0.7279], [0.1707285714, 0.2919380952, 
  0.779247619], [0.1252714286, 0.3242428571, 0.8302714286], 
 [0.0591333333, 0.3598333333, 0.8683333333], [0.0116952381, 0.3875095238, 
  0.8819571429], [0.0059571429, 0.4086142857, 0.8828428571], 
 [0.0165142857, 0.4266, 0.8786333333], [0.032852381, 0.4430428571, 
  0.8719571429], [0.0498142857, 0.4585714286, 0.8640571429], 
 [0.0629333333, 0.4736904762, 0.8554380952], [0.0722666667, 0.4886666667, 
  0.8467], [0.0779428571, 0.5039857143, 0.8383714286], 
 [0.079347619, 0.5200238095, 0.8311809524], [0.0749428571, 0.5375428571, 
  0.8262714286], [0.0640571429, 0.5569857143, 0.8239571429], 
 [0.0487714286, 0.5772238095, 0.8228285714], [0.0343428571, 0.5965809524, 
  0.819852381], [0.0265, 0.6137, 0.8135], [0.0238904762, 0.6286619048, 
  0.8037619048], [0.0230904762, 0.6417857143, 0.7912666667], 
 [0.0227714286, 0.6534857143, 0.7767571429], [0.0266619048, 0.6641952381, 
  0.7607190476], [0.0383714286, 0.6742714286, 0.743552381], 
 [0.0589714286, 0.6837571429, 0.7253857143], 
 [0.0843, 0.6928333333, 0.7061666667], [0.1132952381, 0.7015, 0.6858571429], 
 [0.1452714286, 0.7097571429, 0.6646285714], [0.1801333333, 0.7176571429, 
  0.6424333333], [0.2178285714, 0.7250428571, 0.6192619048], 
 [0.2586428571, 0.7317142857, 0.5954285714], [0.3021714286, 0.7376047619, 
  0.5711857143], [0.3481666667, 0.7424333333, 0.5472666667], 
 [0.3952571429, 0.7459, 0.5244428571], [0.4420095238, 0.7480809524, 
  0.5033142857], [0.4871238095, 0.7490619048, 0.4839761905], 
 [0.5300285714, 0.7491142857, 0.4661142857], [0.5708571429, 0.7485190476, 
  0.4493904762], [0.609852381, 0.7473142857, 0.4336857143], 
 [0.6473, 0.7456, 0.4188], [0.6834190476, 0.7434761905, 0.4044333333], 
 [0.7184095238, 0.7411333333, 0.3904761905], 
 [0.7524857143, 0.7384, 0.3768142857], [0.7858428571, 0.7355666667, 
  0.3632714286], [0.8185047619, 0.7327333333, 0.3497904762], 
 [0.8506571429, 0.7299, 0.3360285714], [0.8824333333, 0.7274333333, 0.3217], 
 [0.9139333333, 0.7257857143, 0.3062761905], [0.9449571429, 0.7261142857, 
  0.2886428571], [0.9738952381, 0.7313952381, 0.266647619], 
 [0.9937714286, 0.7454571429, 0.240347619], [0.9990428571, 0.7653142857, 
  0.2164142857], [0.9955333333, 0.7860571429, 0.196652381], 
 [0.988, 0.8066, 0.1793666667], [0.9788571429, 0.8271428571, 0.1633142857], 
 [0.9697, 0.8481380952, 0.147452381], [0.9625857143, 0.8705142857, 0.1309], 
 [0.9588714286, 0.8949, 0.1132428571], [0.9598238095, 0.9218333333, 
  0.0948380952], [0.9661, 0.9514428571, 0.0755333333], 
 [0.9763, 0.9831, 0.0538]]

parula_map = LinearSegmentedColormap.from_list('parula', cm_data)

In [ ]:
slice_x = slice(30, 162)
slice_y = slice(23, 174)
slice_z = 80

import SimpleITK as sitk

img = sitk.ReadImage(r'path\to\mask\wat_mat_filt_abs_mask.nrrd')
data = sitk.GetArrayFromImage(img)  # NumPy array, note: axis order is (z, y, x)
# mask = mask[::-1,::-1,0]
mask = data[:,0, :]
mask = np.swapaxes(mask, 0, 1)

mat = loadmat(r'path\to\matlab\res_invivo\outParams_img_filt_wat.mat')
wat_mat_filt = mat['outParams'][0][0][0][0][0][1][:,:]
wat_mat_filt = wat_mat_filt[slice_x, slice_y]*mask
fm_mat_filt = mat['outParams']['fieldmap'][0][0][:,:]
fm_mat_filt = fm_mat_filt[slice_x, slice_y]
r2_mat_filt = mat['outParams']['r2starmap'][0][0][:,:]
r2_mat_filt = r2_mat_filt[slice_x, slice_y]
mat = loadmat(r'path\to\matlab\res_invivo\outParams_img_filt_fat.mat')
fat_mat_filt = mat['outParams'][0][0][0][0][1][1][:,:]
fat_mat_filt = fat_mat_filt[slice_x, slice_y]*mask
PDFF_filt = abs(fat_mat_filt) / (abs(fat_mat_filt) + abs(wat_mat_filt))
PDFF_filt = np.nan_to_num(PDFF_filt, nan=0.0)

with h5py.File(r'\path\to\moba_invivo\us1\invivo_20k_spokes_ini_3x1x5_us1_TE_us_1_alph_1_redu_15_TVw_08_TVf_08.h5', 'r') as f:
    print(f.keys())
    img_moba_0 = f['final_images'][:].squeeze()
    img_moba_0 = img_moba_0[:,::-1, ::-1, ]
    img_moba_0 = img_moba_0[:,slice_x, slice_y, slice_z]
    img_moba_0[0,...] = abs(img_moba_0[0,...])*mask
    img_moba_0[1,...] = abs(img_moba_0[1,...])*mask
    img_moba_0[3,...] = abs(img_moba_0[1,...]) / (abs(img_moba_0[1,...]) + abs(img_moba_0[0,...]))
    img_moba_0[3,...] = np.nan_to_num(img_moba_0[3,...], nan=0.0)
    img_moba_0[2,...] = img_moba_0[2,...]*mask
    img_moba_0[3,...] = img_moba_0[3,...]*mask
    
    print(img_moba_0.shape)

with h5py.File(r'\path\to\moba_invivo\us3\invivo_20k_spokes_ini_3x2x10_us3_TE_us_1_alph_1_redu_15_TVw_08_TVf_08.h5', 'r') as f:
    print(f.keys())
    img_moba_1 = f['final_images'][:].squeeze()
    img_moba_1 = img_moba_1[:,::-1, ::-1, ]
    img_moba_1 = img_moba_1[:,slice_x, slice_y, slice_z]
    img_moba_1[0,...] = abs(img_moba_1[0,...])*mask
    img_moba_1[1,...] = abs(img_moba_1[1,...])*mask
    img_moba_1[3,...] = abs(img_moba_1[1,...]) / (abs(img_moba_1[1,...]) + abs(img_moba_1[0,...]))
    img_moba_1[3,...] = np.nan_to_num(img_moba_1[3,...], nan=0.0)
    
    print(img_moba_1.shape)

with h5py.File(r'\path\to\moba_invivo\us4\invivo_20k_spokes_ini_3x2x10_us4_TE_us_1_alph_1_redu_15_TVw_08_TVf_08.h5', 'r') as f:
    print(f.keys())
    img_moba_2 = f['final_images'][:].squeeze()
    img_moba_2 = img_moba_2[:,::-1, ::-1, slice_z]
    img_moba_2 = img_moba_2[:,slice_x, slice_y, ]
    img_moba_2[0,...] = abs(img_moba_2[0,...])*mask
    img_moba_2[1,...] = abs(img_moba_2[1,...])*mask
    img_moba_2[3,...] = abs(img_moba_2[1,...]) / (abs(img_moba_2[1,...]) + abs(img_moba_2[0,...]))
    img_moba_2[3,...] = np.nan_to_num(img_moba_2[3,...], nan=0.0)   
    
    print(img_moba_2.shape)



In [ ]:
wat_perc_org = 95
fat_perc_org = 85

fat_perc_moba = 83
wat_perc_moba = 95

PDFF_max = 1

In [ ]:
N_cols = 3
N_rows = 5
i = 0
plt.figure(figsize=(10, 18))

i += 1
ax = plt.subplot(N_rows, N_cols, i)
plt.imshow(abs(wat_mat_filt), cmap = 'gray', vmax=np.percentile(abs(wat_mat_filt), wat_perc_org))
ax.set_xticks([])
ax.set_yticks([])
ax.set_ylabel('NUFFT.H filt 20k', fontsize=14)
ax.set_title('Water', fontsize=14)
i += 1

plt.subplot(N_rows, N_cols, i)
plt.imshow(abs(fat_mat_filt), cmap = 'gray', vmax=np.percentile(abs(fat_mat_filt), fat_perc_org))
plt.title('Fat', fontsize=14)
plt.axis('off')
i += 1

plt.subplot(N_rows, N_cols, i)
plt.imshow(np.real(PDFF_filt), cmap = parula_map, vmin=0, vmax=PDFF_max)
plt.axis('off')
plt.title('PDFF', fontsize=14)
i += 1

ax = plt.subplot(N_rows, N_cols, i)
plt.imshow(abs(img_moba_0[0,...]), cmap = 'gray', vmax=np.percentile(abs(img_moba_0[0,...]), wat_perc_moba))
ax.set_xticks([])
ax.set_yticks([])
ax.set_ylabel('MoBa 20k us1', fontsize=14)
i += 1

plt.subplot(N_rows, N_cols, i)
plt.imshow(abs(img_moba_0[1,...]), cmap = 'gray', vmax=np.percentile(abs(img_moba_0[1,...]), fat_perc_moba))
plt.axis('off')
i += 1

plt.subplot(N_rows, N_cols, i)
plt.imshow(np.real(img_moba_0[3,...]), cmap = parula_map, vmin=0, vmax=PDFF_max)
plt.axis('off')
i += 1

ax = plt.subplot(N_rows, N_cols, i)
plt.imshow(abs(img_moba_1[0,...]), cmap = 'gray', vmax=np.percentile(abs(img_moba_1[0,...]), wat_perc_moba))
ax.set_xticks([])
ax.set_yticks([])
ax.set_ylabel('MoBa 20k us3', fontsize=14)
i += 1

plt.subplot(N_rows, N_cols, i)
plt.imshow(abs(img_moba_1[1,...]), cmap = 'gray', vmax=np.percentile(abs(img_moba_1[1,...]), fat_perc_moba))
plt.axis('off')
i += 1

plt.subplot(N_rows, N_cols, i)
plt.imshow(np.real(img_moba_1[3,...]), cmap = parula_map, vmin=0, vmax=PDFF_max)
plt.axis('off')
i += 1


ax = plt.subplot(N_rows, N_cols, i)
plt.imshow(abs(img_moba_2[0,...]), cmap = 'gray', vmax=np.percentile(abs(img_moba_2[0,...]), wat_perc_moba))
ax.set_xticks([])
ax.set_yticks([])
ax.set_ylabel('MoBa 20k us3.5', fontsize=14)
i += 1

plt.subplot(N_rows, N_cols, i)
plt.imshow(abs(img_moba_2[1,...]), cmap = 'gray', vmax=np.percentile(abs(img_moba_2[1,...]), fat_perc_moba))
plt.axis('off')
i += 1

plt.subplot(N_rows, N_cols, i)
plt.imshow(np.real(img_moba_2[3,...]), cmap = parula_map, vmin=0, vmax=PDFF_max)
plt.axis('off')
i += 1


plt.tight_layout()